# 🔥 Train Multi-Task CE — M2-CE: Không dùng EDL, Không Dual-Branch
**Thực nghiệm M2-CE:** Multi-Task Learning dùng **Cross-Entropy thuần (Softmax)** cho nhánh phân loại.  
Đây là ablation study: loại bỏ EDL để so sánh với M2-EDL và M3.

| Thuộc tính | Giá trị |
|---|---|
| Phân loại loss | CrossEntropyLoss (Softmax) |
| Phân đoạn loss | Focal Tversky |
| Nhánh viền (Dual-Branch) | ❌ Tắt (`--no_boundary`) |
| EDL | ❌ Tắt (`--no_edl`) |
| Run name | `run_multitask_pure_ce` |

### ⚙️ BƯỚC 1: Kiểm tra GPU
*(Chọn `Runtime` → `Change runtime type` → `T4 GPU` hoặc `A100`)*

In [ ]:
!nvidia-smi

### 📁 BƯỚC 2: Kết nối Google Drive & Thiết lập Môi trường Dự án

In [ ]:
from google.colab import drive
import os

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Thiết lập thư mục dự án và đồng bộ code nhánh feat/mt-eanet-edl
if not os.path.exists('/content/skin-burn-analysis-system'):
    %cd /content
    print('Lần đầu chạy: Đang clone dự án từ GitHub nhánh feat/mt-eanet-edl...')
    !git clone -b feat/mt-eanet-edl https://github.com/Triminh15/skin-burn-analysis-system.git
    %cd /content/skin-burn-analysis-system
else:
    print('✅ Thư mục skin-burn-analysis-system đã tồn tại trên Colab.')
    print('Đang cập nhật/đồng bộ code mới nhất từ nhánh feat/mt-eanet-edl...')
    %cd /content/skin-burn-analysis-system
    !git init 2>/dev/null || true
    !git remote set-url origin https://github.com/Triminh15/skin-burn-analysis-system.git 2>/dev/null || git remote add origin https://github.com/Triminh15/skin-burn-analysis-system.git
    !git fetch origin feat/mt-eanet-edl
    !git checkout -f feat/mt-eanet-edl
    !git pull origin feat/mt-eanet-edl

# 3. Xác nhận thư mục làm việc
%cd /content/skin-burn-analysis-system
!pwd
!git log --oneline -3

### 📦 BƯỚC 3: Cài đặt Thư viện Phụ thuộc

In [ ]:
%cd /content/skin-burn-analysis-system
!pip install -q timm albumentations segmentation-models-pytorch matplotlib pandas tqdm scikit-learn
print('✅ Thư viện đã cài xong!')

### 🗄️ BƯỚC 4: Giải nén Dữ liệu (`data_processed.zip`)

In [ ]:
%cd /content/skin-burn-analysis-system 2>/dev/null || %cd /content/burn-analysis 2>/dev/null || true
import os
import zipfile
import shutil

candidate_zips = [
    '/content/drive/MyDrive/processed.zip',
    '/content/drive/MyDrive/data_processed.zip',
    '/content/drive/MyDrive/skin_burn/data_processed.zip',
    '/content/drive/MyDrive/skin_burn/processed.zip',
    '/content/drive/MyDrive/Do_An_tri/data_processed.zip',
    '/content/drive/MyDrive/Do_An_tri/processed.zip',
    '/content/processed.zip',
    '/content/data_processed.zip',
    'processed.zip',
    'data_processed.zip'
]

zip_path = None
for p in candidate_zips:
    if os.path.exists(p):
        zip_path = p
        break

# Tự động quét sâu trong Drive nếu chưa thấy ở các đường dẫn thông thường
if not zip_path and os.path.exists('/content/drive/MyDrive'):
    print('🔍 Đang tự động quét tìm file zip dữ liệu trong Google Drive...')
    for root, dirs, files_in_dir in os.walk('/content/drive/MyDrive'):
        if '.Trash' in root or '.shortcut-targets-by-id' in root:
            continue
        for file_name in files_in_dir:
            if file_name.endswith('.zip') and any(k in file_name.lower() for k in ['processed', 'data_processed', 'skin_burn']):
                zip_path = os.path.join(root, file_name)
                break
        if zip_path:
            break

# Nếu Drive mới chưa có file zip, mở upload trực tiếp từ máy tính lên Colab
if not zip_path:
    print('⚠️ Chưa tìm thấy file zip trên Google Drive của tài khoản này!')
    print('👉 Đang mở hộp thoại để bạn chọn upload data_processed.zip từ máy tính lên Colab (31MB chỉ mất 5-10s)...')
    try:
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.zip'):
                zip_path = fn
                # Tự động lưu sang Google Drive mới để lần sau không cần upload lại
                if os.path.exists('/content/drive/MyDrive'):
                    shutil.copy(fn, os.path.join('/content/drive/MyDrive', fn))
                    print(f'💾 Đã tự động lưu {fn} vào Drive mới để dùng cho những lần sau!')
                break
    except Exception as e:
        print('Lưu ý upload:', e)

if not zip_path or not os.path.exists(zip_path):
    print('❌ Vẫn chưa có file zip dữ liệu để giải nén!')
else:
    print(f'📦 Đang giải nén {zip_path} bằng Python (tự động chuẩn hóa dấu gạch chéo Windows)...')
    os.makedirs('data/processed', exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for member in zf.infolist():
            clean_name = member.filename.replace('\\', '/').strip('/')
            if member.is_dir() or clean_name.endswith('/'):
                continue
            if 'images/' in clean_name:
                sub_rel = 'images/' + clean_name.split('images/')[-1]
            elif 'masks/' in clean_name:
                sub_rel = 'masks/' + clean_name.split('masks/')[-1]
            elif 'rois/' in clean_name:
                sub_rel = 'rois/' + clean_name.split('rois/')[-1]
            else:
                sub_rel = clean_name
            dest_path = os.path.join('data/processed', sub_rel)
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            with zf.open(member) as src, open(dest_path, 'wb') as dst:
                dst.write(src.read())
    print('✅ Giải nén và định tuyến thư mục hoàn tất!')

n_img = len(os.listdir('data/processed/images')) if os.path.exists('data/processed/images') else 0
n_mask = len(os.listdir('data/processed/masks')) if os.path.exists('data/processed/masks') else 0
n_roi = len(os.listdir('data/processed/rois')) if os.path.exists('data/processed/rois') else 0

print(f'[Dữ liệu] Ảnh gốc (data/processed/images): {n_img} ảnh')
print(f'[Dữ liệu] Mặt nạ  (data/processed/masks):  {n_mask} ảnh')
print(f'[Dữ liệu] ROIs    (data/processed/rois):   {n_roi} mục')
print('[Splits] train.csv:', os.path.exists('data/splits/train.csv'))
print('[Splits] val.csv  :', os.path.exists('data/splits/val.csv'))
print('[Splits] test.csv :', os.path.exists('data/splits/test.csv'))

---
## 🚀 BƯỚC 5: TRAIN M2-CE — Multi-Task (Cross-Entropy / Softmax, Không EDL)
- **`--no_edl`** → Dùng CrossEntropyLoss thay vì EDL
- **`--no_boundary`** → Tắt nhánh Dual-Branch viền ban đỏ
- **`--cls_loss ce`** → Chỉ định rõ ràng dùng Cross-Entropy

In [ ]:
%cd /content/skin-burn-analysis-system 2>/dev/null || %cd /content/burn-analysis 2>/dev/null || true
!python src/vision/train_multitask.py \
    --model_type multitask_edl \
    --no_boundary \
    --w_bnd 0.0 \
    --no_edl \
    --cls_loss ce \
    --seg_loss combo \
    --w_seg 2.0 \
    --w_cls 0.15 \
    --lr 1e-4 \
    --dec_lr_mult 3.0 \
    --cls_lr_mult 5.0 \
    --epochs 40 \
    --patience 12 \
    --batch_size 16 \
    --output_dir outputs/checkpoints/multitask \
    --run-name run_multitask_pure_ce \
    --seed 42

---
## 📊 BƯỚC 6: Xem Kết quả Test & Bảng So Sánh

In [ ]:
%cd /content/skin-burn-analysis-system
import json
import pandas as pd
import os

# ---- Kết quả M2-CE ----
result_path = 'outputs/checkpoints/multitask/run_multitask_pure_ce/test_results.json'

if not os.path.exists(result_path):
    print(f'❌ Không tìm thấy: {result_path}')
    print('   Hãy kiểm tra lại Bước 5 đã chạy thành công chưa.')
else:
    with open(result_path, 'r', encoding='utf-8') as f:
        r = json.load(f)

    dice    = r.get('dice', 0.0)
    iou     = r.get('iou', 0.0)
    px_acc  = r.get('pixel_accuracy', 0.0)
    cls_acc = r.get('accuracy', r.get('test_accuracy', 0.0))
    macro_f1= r.get('macro_f1', r.get('test_macro_f1', 0.0))
    rec3    = r.get('recall_degree_3', r.get('test_recall_degree_3', 0.0))
    loss_v  = r.get('loss_total', r.get('test_loss', None))

    print('=' * 65)
    print('  📋 KẾT QUẢ: M2-CE — Multi-Task Pure (CrossEntropy / Softmax)')
    print('=' * 65)
    print(f'  🎯 Phân đoạn:')
    print(f'     Test Dice      : {dice:.2%}')
    print(f'     Test IoU       : {iou:.2%}')
    print(f'     Pixel Accuracy : {px_acc:.2%}')
    if loss_v is not None:
        print(f'     Test Loss      : {loss_v:.4f}')
    print(f'  🏷️  Phân loại (CE / Softmax):')
    print(f'     Cls Accuracy   : {cls_acc:.2%}')
    print(f'     Macro F1       : {macro_f1:.2%}')
    print(f'     Recall Độ 3    : {rec3:.2%}')
    print(f'  ℹ️  Params: ~24.48M | Latency: ~15ms | ROI crop: Không cần')
    print('=' * 65)

    # ---- Bảng so sánh với Baseline Bảng #01 ----
    baseline_rows = [
        {'Model ID': 'U-Net (ResNet-34) + ComboLoss (Bảng #01)',       'Kiến trúc': 'Đơn nhiệm (Seg Only)', 'Test Dice': '65.19%', 'Test IoU': '48.36%', 'Test Loss': '0.3522', 'Pixel Acc': '86.86%', 'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A', 'Bất định (u)': 'N/A', 'Hội chẩn (u>=0.4)': 'N/A', 'Params': '24.44M', 'Latency': '~15ms', 'Cắt ROI?': 'Bắt buộc'},
        {'Model ID': 'U-Net (ResNet-34) + Focal Tversky (Bảng #01)',   'Kiến trúc': 'Đơn nhiệm (Seg Only)', 'Test Dice': '65.07%', 'Test IoU': '48.23%', 'Test Loss': '0.2012', 'Pixel Acc': '85.17%', 'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A', 'Bất định (u)': 'N/A', 'Hội chẩn (u>=0.4)': 'N/A', 'Params': '24.44M', 'Latency': '~15ms', 'Cắt ROI?': 'Bắt buộc'},
        {'Model ID': 'U-Net (ConvNeXt-T) + Focal Tversky (Bảng #01)', 'Kiến trúc': 'Đơn nhiệm (Seg Only)', 'Test Dice': '63.60%', 'Test IoU': '46.63%', 'Test Loss': '0.2111', 'Pixel Acc': '83.74%', 'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A', 'Bất định (u)': 'N/A', 'Hội chẩn (u>=0.4)': 'N/A', 'Params': '31.93M', 'Latency': '~15ms', 'Cắt ROI?': 'Bắt buộc'},
        {'Model ID': 'Attention U-Net++ (ConvNeXt-T) (Bảng #01)',     'Kiến trúc': 'Đơn nhiệm (Seg Only)', 'Test Dice': '61.57%', 'Test IoU': '44.48%', 'Test Loss': '0.2200', 'Pixel Acc': '82.08%', 'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A', 'Bất định (u)': 'N/A', 'Hội chẩn (u>=0.4)': 'N/A', 'Params': '32.14M', 'Latency': '~15ms', 'Cắt ROI?': 'Bắt buộc'},
        # M2-EDL và M3 (kết quả đã có từ notebook EDL)
        {'Model ID': 'M2-EDL: Multi-Task Pure (Có EDL)',               'Kiến trúc': 'Đa nhiệm EDL',        'Test Dice': '63.81%', 'Test IoU': '46.85%', 'Test Loss': 'N/A',    'Pixel Acc': '83.69%', 'Cls Acc': '75.40%', 'Macro F1': '74.21%', 'Recall Độ 3': '67.65%', 'Bất định (u)': '0.4932', 'Hội chẩn (u>=0.4)': '55.61%', 'Params': '24.48M', 'Latency': '~15ms', 'Cắt ROI?': 'Không cần'},
        {'Model ID': 'M3: MT-EANet-EDL (Dual-Branch)',                 'Kiến trúc': 'Đa nhiệm Dual-Branch','Test Dice': '60.44%', 'Test IoU': '43.31%', 'Test Loss': 'N/A',    'Pixel Acc': '80.14%', 'Cls Acc': '74.33%', 'Macro F1': '73.05%', 'Recall Độ 3': '64.71%', 'Bất định (u)': '0.5224', 'Hội chẩn (u>=0.4)': '64.17%', 'Params': '24.50M', 'Latency': '~16ms', 'Cắt ROI?': 'Không cần'},
    ]

    ce_row = {
        'Model ID': 'M2-CE: Multi-Task Pure (Không EDL) ⭐',
        'Kiến trúc': 'Đa nhiệm CE (Softmax)',
        'Test Dice':       f'{dice:.2%}',
        'Test IoU':        f'{iou:.2%}',
        'Test Loss':       f'{loss_v:.4f}' if loss_v is not None else 'N/A',
        'Pixel Acc':       f'{px_acc:.2%}',
        'Cls Acc':         f'{cls_acc:.2%}',
        'Macro F1':        f'{macro_f1:.2%}',
        'Recall Độ 3':     f'{rec3:.2%}',
        'Bất định (u)':    'N/A',
        'Hội chẩn (u>=0.4)': 'N/A',
        'Params': '24.48M', 'Latency': '~15ms', 'Cắt ROI?': 'Không cần',
    }

    df = pd.DataFrame(baseline_rows + [ce_row])
    print('\n' + '=' * 115)
    print('🏆 BẢNG SO SÁNH 7 MÔ HÌNH (BASELINE + M2-CE + M2-EDL + M3)')
    print('=' * 115)
    display(df)

    os.makedirs('outputs', exist_ok=True)
    df.to_csv('outputs/full_model_comparison_with_ce.csv', index=False, encoding='utf-8')
    print('💾 Đã lưu: outputs/full_model_comparison_with_ce.csv')

### 💾 BƯỚC 7: Sao lưu Checkpoint & Kết quả vào Google Drive

In [ ]:
%cd /content/skin-burn-analysis-system
!mkdir -p /content/drive/MyDrive/burn_research_outputs
!cp -r outputs/checkpoints/* /content/drive/MyDrive/burn_research_outputs/
!cp -f outputs/full_model_comparison_with_ce.csv /content/drive/MyDrive/burn_research_outputs/ 2>/dev/null || true
print('✅ Toàn bộ checkpoint và bảng so sánh đã backup vào: /content/drive/MyDrive/burn_research_outputs/')